In [2]:
import pandas as pd

INPUT_FILE = "data\processed\incubator_stage3_targets.csv"

df = pd.read_csv(INPUT_FILE)

print(f"Dataset shape: {df.shape}")

<>:3: SyntaxWarning: invalid escape sequence '\p'
<>:3: SyntaxWarning: invalid escape sequence '\p'
C:\Users\Fady\AppData\Local\Temp\ipykernel_5560\457765014.py:3: SyntaxWarning: invalid escape sequence '\p'
  INPUT_FILE = "data\processed\incubator_stage3_targets.csv"


Dataset shape: (53061, 41)


In [3]:
TARGET_COLUMNS = [
    "temperature_risk_5m",
    "temperature_risk_10m",
    "temperature_risk_15m",
]

# All three future targets must exist
df = df.dropna(subset=TARGET_COLUMNS).copy()

DROP_FROM_FEATURES = [
    "record_id",
    "sequence_id",
    "estimated_timestamp",
    "target_segment_id",

    # Future temperature values — DATA LEAKAGE if used as inputs
    "target_avg_t_5m",
    "target_avg_t_10m",
    "target_avg_t_15m",

    # Outputs
    "temperature_risk_5m",
    "temperature_risk_10m",
    "temperature_risk_15m",
]

X = df.drop(columns=DROP_FROM_FEATURES)
Y = df[TARGET_COLUMNS]

print(f"Usable rows: {len(df)}")
print(f"Number of input features: {X.shape[1]}")
print(f"Number of outputs: {Y.shape[1]}")

Usable rows: 52330
Number of input features: 31
Number of outputs: 3


In [4]:
print("\nTarget class distributions:")

for target in TARGET_COLUMNS:
    print(f"\n{target}")
    print(Y[target].value_counts())
    print((Y[target].value_counts(normalize=True) * 100).round(2))


Target class distributions:

temperature_risk_5m
temperature_risk_5m
NORMAL    48701
LOW        2676
HIGH        953
Name: count, dtype: int64
temperature_risk_5m
NORMAL    93.07
LOW        5.11
HIGH       1.82
Name: proportion, dtype: float64

temperature_risk_10m
temperature_risk_10m
NORMAL    46533
LOW        4159
HIGH       1638
Name: count, dtype: int64
temperature_risk_10m
NORMAL    88.92
LOW        7.95
HIGH       3.13
Name: proportion, dtype: float64

temperature_risk_15m
temperature_risk_15m
NORMAL    44502
LOW        5558
HIGH       2270
Name: count, dtype: int64
temperature_risk_15m
NORMAL    85.04
LOW       10.62
HIGH       4.34
Name: proportion, dtype: float64


In [ ]:
# ---------------------------------------------------------
# Chronological split by continuous sequences
# ---------------------------------------------------------

TRAIN_SEQUENCES = [1, 2, 3, 5]
VAL_SEQUENCES = [7]
TEST_SEQUENCES = [8, 9]

# Keep acquisition order
df = df.sort_values("record_id").reset_index(drop=True)

train_df = df[df["sequence_id"].isin(TRAIN_SEQUENCES)].copy()
val_df = df[df["sequence_id"].isin(VAL_SEQUENCES)].copy()
test_df = df[df["sequence_id"].isin(TEST_SEQUENCES)].copy()

print("\nSequence-based split:")
print(f"Train:      {len(train_df)} rows - sequences {TRAIN_SEQUENCES}")
print(f"Validation: {len(val_df)} rows - sequences {VAL_SEQUENCES}")
print(f"Test:       {len(test_df)} rows - sequences {TEST_SEQUENCES}")

# ---------------------------------------------------------
# Build Train / Validation / Test feature and target sets
# ---------------------------------------------------------

FEATURE_COLUMNS = X.columns.tolist()

X_train = train_df[FEATURE_COLUMNS].copy()
Y_train = train_df[TARGET_COLUMNS].copy()

X_val = val_df[FEATURE_COLUMNS].copy()
Y_val = val_df[TARGET_COLUMNS].copy()

X_test = test_df[FEATURE_COLUMNS].copy()
Y_test = test_df[TARGET_COLUMNS].copy()

print("\nML dataset shapes:")
print(f"X_train: {X_train.shape} | Y_train: {Y_train.shape}")
print(f"X_val:   {X_val.shape} | Y_val:   {Y_val.shape}")
print(f"X_test:  {X_test.shape} | Y_test:  {Y_test.shape}")


# ---------------------------------------------------------
# Check class distributions in each split
# ---------------------------------------------------------

EXPECTED_CLASSES = {"LOW", "NORMAL", "HIGH"}

splits = {
    "TRAIN": Y_train,
    "VALIDATION": Y_val,
    "TEST": Y_test,
}

for split_name, y_split in splits.items():

    print(f"\n--- {split_name} ---")

    for target in TARGET_COLUMNS:

        counts = y_split[target].value_counts()
        percentages = (
            y_split[target]
            .value_counts(normalize=True)
            .mul(100)
            .round(2)
        )

        print(f"\n{target}")
        print(counts)
        print("Percentages:")
        print(percentages)

        missing_classes = EXPECTED_CLASSES - set(counts.index)

        if missing_classes:
            raise ValueError(
                f"{split_name} - {target} missing classes: "
                f"{missing_classes}"
            )


